<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/06_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 06 — Multi-Agent Hierarchies

Module 05 gave you workflow agents — **named, declarative** control flow. Sequential, Parallel, Loop. Use them when you can name the workflow.

This module gives you the other half: **LLM-driven** control flow. When the user's input decides which specialist runs, or when a coordinator should call a specialist like a function, you don't want a workflow agent. You want the LLM to route.

ADK has two patterns for this. They look superficially similar. They behave very differently once you pay attention to who is in charge of the conversation.

- **`sub_agents`** — the coordinator *transfers* control to a specialist. The specialist owns the conversation. The org-chart pattern.
- **`AgentTool`** — the coordinator *calls* a specialist like a function. The coordinator stays in charge. The consultant pattern.

By the end of this notebook you'll have built the same coordinator-plus-two-specialists team both ways. The same input produces the same answer, but the event stream looks noticeably different. Which pattern is right depends on whether the specialist should drive the conversation or hand the mic back.

**Interlude:** at the end, two minutes on **multi-agent decomposition** — Pattern 8 from *Agentic Design Patterns* — on when multi-agent architectures pay for their coordination tax and when they don't.

**Running cost:** under $0.01.

# Setup

In [1]:
!pip install -q google-adk==2.4.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/google/gemini-2.5-flash-lite"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/google/gemini-2.5-flash-lite


## Imports

In [3]:
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.agent_tool import AgentTool
from google.genai import types

print("✅ Imports successful.")

09:18:36 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


09:18:36 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✅ Imports successful.


# The Composition Question, Recap

You have three ways to combine agents in ADK:

| Pattern | Control flow | When |
|---|---|---|
| **Workflow agent** (M05) | Declarative — Sequential, Parallel, Loop | You can name the workflow |
| **`sub_agents`** (this module) | LLM decides — via transfer | User input shapes the flow; specialist should own the dialog |
| **`AgentTool`** (this module) | LLM decides — via function call | User input shapes the flow; coordinator stays in charge |

The `sub_agents` and `AgentTool` patterns both let the LLM decide what runs next. The difference is **who is in charge afterward.**

# Pattern 1 — `sub_agents`: the Org-Chart Transfer

The coordinator has a list of children in its `sub_agents=` parameter. When the coordinator's LLM decides a specialist is more appropriate, it emits a structured call to a built-in tool called `transfer_to_agent(agent_name=...)`. ADK catches that call and **transfers control to the named child**. The child runs, produces a response, and *the child's response is what the user sees*.

Think of it as an org chart. The coordinator is the manager; it reads the incoming ticket and routes it. Once routed, the specialist owns the conversation — for this turn, and for future turns in the same session, unless transferred again.

The mechanism ADK provides:
- The `transfer_to_agent` tool is **automatically available** to any agent that has `sub_agents=`. You don't register it; ADK injects it.
- The child's `description=` is what the coordinator's LLM reads to decide whether to route. Write descriptions carefully — that's the routing schema.

In [4]:
# Two specialists — greeter and weather_specialist.
greeter = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles greetings and casual chat.",
    instruction="You are a friendly greeter. Respond warmly in one short sentence.",
)

weather_specialist = LlmAgent(
    name="weather_specialist",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles weather questions for any city.",
    instruction=(
        "You are a weather assistant. You do not have real weather tools; "
        "produce plausible one-sentence weather reports for the city asked about."
    ),
)

# The coordinator routes via sub_agents.
coordinator_subagents = LlmAgent(
    name="coordinator_subagents",
    model=LiteLlm(model=MODEL_STRING),
    description="Coordinator that routes to specialists via transfer.",
    instruction="""You coordinate a small team. You have two specialists available:
- `greeter` for greetings and casual chat.
- `weather_specialist` for weather questions.

When the user's message fits one of them, use transfer_to_agent to hand over.
Do not answer greetings or weather questions yourself — always delegate.""",
    sub_agents=[greeter, weather_specialist],
)

print("✅ Coordinator (sub_agents pattern) ready.")

✅ Coordinator (sub_agents pattern) ready.


In [5]:
APP = "m06_demos"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, session_id: str = None):
    sid = session_id or f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:200]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[{ev.author}] → {p.function_call.name}({args})")
                if p.function_response:
                    resp = str(p.function_response.response)
                    if len(resp) < 200:
                        print(f"[tool_resp] {resp}")

print("✅ chat() ready.")

✅ chat() ready.


## Demo — transfer routing

In [6]:
await chat(coordinator_subagents, "Hi there, how are you?")

USER: Hi there, how are you?



[coordinator_subagents] → transfer_to_agent({'agent_name': 'greeter'})
[tool_resp] {'result': None}


[greeter] Hello there! I'm doing great, thanks for asking. How can I help you today?


In [7]:
await chat(coordinator_subagents, "What's the weather in Prague?")

USER: What's the weather in Prague?



[coordinator_subagents] → transfer_to_agent({'agent_name': 'weather_specialist'})
[tool_resp] {'result': None}


[weather_specialist] The weather in Prague is currently partly cloudy with a gentle breeze.


Two observations from the event stream.

- The coordinator emitted `transfer_to_agent(agent_name='...')` as a tool call. ADK caught it and routed to the named child.
- The child's output appears under its own author — `[greeter]` and `[weather_specialist]`, not `[coordinator_subagents]`. **The child produced the final response.** That's the transfer pattern.

One subtle production fact: after a transfer, the child remains the active agent for the rest of the session by default. If the user asks a follow-up question in the same session, it goes to the specialist, not back to the coordinator. The coordinator can regain control by having its own logic (or the child can transfer back). M03's session state applies here — the active-agent is part of the session.

# Pattern 2 — `AgentTool`: the Consultant Pattern

Same team, different wiring. Instead of putting the specialists in `sub_agents=`, we wrap each as an `AgentTool` and put them in `tools=`. Now from the coordinator's LLM's perspective, the specialists are **tools**, not transfer targets.

When the coordinator decides to use a specialist, it emits a regular function call — same mechanism as calling `get_weather` or `search_tickets`. ADK executes the specialist, gets its output, and hands it back as a tool-response event. **The coordinator reads the response and produces the final user-facing reply itself.** The specialist never gets the mic.

In [8]:
# Same specialists; same descriptions; different wiring.
# Use separate instances so they don't accidentally share parent state.
greeter_t = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles greetings and casual chat. Input: user message. Output: one warm sentence.",
    instruction="Respond warmly in one short sentence.",
)

weather_specialist_t = LlmAgent(
    name="weather_specialist",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports weather for any city. Input: the city name. Output: one-sentence weather report.",
    instruction="Produce a plausible one-sentence weather report.",
)

coordinator_tools = LlmAgent(
    name="coordinator_tools",
    model=LiteLlm(model=MODEL_STRING),
    description="Coordinator that calls specialists as tools.",
    instruction="""You have exactly two tools available, and you must use one of them
for every user message:

- Call the tool named `greeter` for greetings, small talk, or casual chat.
- Call the tool named `weather_specialist` for any weather question.

Do NOT invent other tool names. Do NOT answer directly. Always call one of
those two tools, pass it the user's message as the `request` argument, read
the result, and wrap it in your own friendly reply.""",
    tools=[
        AgentTool(agent=greeter_t),
        AgentTool(agent=weather_specialist_t),
    ],
)

print("✅ Coordinator (AgentTool pattern) ready.")

✅ Coordinator (AgentTool pattern) ready.


## Demo — consultant calls

In [9]:
await chat(coordinator_tools, "Hi there, how are you?")

USER: Hi there, how are you?



[coordinator_tools] → greeter({'request': 'Hi there, how are you?'})


[tool_resp] {'result': "Hello there! I'm doing wonderfully, thank you for asking!"}


[coordinator_tools] Hello there! I'm doing wonderfully, thank you for asking!


In [10]:
await chat(coordinator_tools, "What's the weather in Prague?")

USER: What's the weather in Prague?



[coordinator_tools] → weather_specialist({'request': 'Prague'})


[tool_resp] {'result': 'A crisp autumn breeze will accompany sunny skies over Prague today, with temperatures peaking around 12 degrees Celsius.'}


[coordinator_tools] The weather in Prague is expected to be sunny with a crisp autumn breeze, and the temperature will reach about 12 degrees Celsius.


Notice the differences in the event stream.

- The coordinator emitted a regular tool call — `weather_specialist(...)` — not `transfer_to_agent`.
- A `[tool_resp]` event carries the specialist's output back.
- The **final response author is `[coordinator_tools]`**, not the specialist. The coordinator read the specialist's output, wrapped it in its own voice, and produced the reply.

This is the consultant pattern. The specialist answered a specific question and stepped back; the coordinator remained in charge.

# Side-by-Side

Same input, same team, two different compositions. The event streams tell you which pattern is in use.

## `sub_agents` — the transfer pattern

```
USER: What's the weather in Prague?

[coordinator_subagents] → transfer_to_agent(agent_name='weather_specialist')
[weather_specialist]    Prague is cloudy and cool, with a gentle breeze.
                        ^^^ final response comes from the specialist
```

## `AgentTool` — the consultant pattern

```
USER: What's the weather in Prague?

[coordinator_tools] → weather_specialist(request='...')
[tool_resp]            {'result': 'In Prague, expect a partly cloudy day...'}
[coordinator_tools]    The weather in Prague is partly cloudy with a gentle breeze.
                       ^^^ final response comes from the coordinator,
                           wrapping the specialist's output
```

**Tell-tale sign:** look at the author of the final response. If it's a specialist, you're looking at a transfer. If it's the coordinator, you're looking at a consultant call.

# When to Pick Which

| Use `sub_agents` (transfer) when... | Use `AgentTool` (consultant) when... |
|---|---|
| The specialist should **own the conversation** after routing | The specialist should **answer one question and step back** |
| The child might take multiple turns with the user before handing back | The child has a clean input/output contract |
| Topic shifts — the specialist is the right conversation partner for the whole topic | The coordinator needs to compose the specialist's output with other sources |
| You want the user to perceive talking to a specialist | You want the user to perceive talking to one assistant that has specialists behind the scenes |

A practical rule: if the specialist's work is part of a **larger answer** that also incorporates other information, use `AgentTool`. If the specialist's work *is* the answer, use `sub_agents`.

### A concrete example for each

**`sub_agents` scenario.** IT support desk with a billing specialist and a hardware specialist. The user says "my laptop won't boot." The coordinator routes to hardware. Over the next five turns of the session, the hardware specialist walks the user through diagnosis. That's an extended conversation the coordinator shouldn't mediate — `sub_agents` is right.

**`AgentTool` scenario.** Coding assistant whose orchestrator wraps a specialist code-analyzer, a specialist docs-lookup, and a specialist test-runner. The user asks "why is my test failing?" The orchestrator calls all three, gets their outputs, synthesizes one answer. Each specialist's contribution is a single structured response; the orchestrator owns composing them. That's `AgentTool`.

# Interlude — Multi-Agent Decomposition

> *From "Agentic Design Patterns," Chapter 8. Two minutes on when to build multi-agent architectures — and when not to.*

Multi-agent architectures are fashionable. They're also expensive. Before you decompose a task into several specialist agents, consider whether the coordination tax is worth paying.

The coordination tax has three parts:

1. **Extra LLM calls.** Every transfer or `AgentTool` call is an additional model invocation. A two-specialist system makes at least two calls per turn. A five-specialist system can make ten or more. This compounds latency and cost.
2. **Routing errors.** Every routing decision is a chance to pick the wrong specialist. The more specialists, the wider the miss surface.
3. **Context fragmentation.** Each child has its own system prompt. Information you put in the coordinator's instruction doesn't automatically reach the children. Specialists sometimes fail because they lack context the coordinator had.

The publication proposes three tests before you decompose:

**Test 1 — Reuse test.** Will any of the specialists be used elsewhere (by a different top-level agent, in a different product)? If yes, decomposing into standalone agents is worth it — the specialists become reusable components. If no, consider keeping everything in one agent with more tools.

**Test 2 — Model-heterogeneity test.** Does each specialist benefit from a different model? If the routing decision is cheap (route on Haiku) but the answer is hard (resolve on Opus), separate agents let you mix models per task. If they all use the same model, the decomposition buys you less.

**Test 3 — Instruction-scale test.** Is a single system prompt getting unmanageable? If you're at 500 lines of instructions with overlapping rules, breaking it up across specialists is a legitimate way to reduce each prompt to its core job. But if your single-agent prompt is 30 lines, don't split it just because you can.

If none of the three tests pass, **keep it to one agent with tools**. A single well-written agent with 8 tools is almost always simpler, cheaper, and faster than a coordinator-plus-specialists architecture doing the same work. Multi-agent is the right answer sometimes; it is not the right answer by default.

# Your Turn

1. **Refactor the transfer demo to use a third specialist.** Add a `joke_specialist` agent. Route with the `sub_agents` pattern. Ask it three questions — a greeting, a weather query, a joke request — and watch the routing.
2. **Refactor the consultant demo to use two tools at once.** Ask the `coordinator_tools` agent: *"Greet me, and tell me the weather in Warsaw."* Does it call both specialists in one turn? Which order?
3. **The three tests, applied.** Take an agent you've built (M02's guarded-ticket agent, say) and apply the three-tests from the interlude. Would decomposing it into multiple agents be justified? Write 2-3 sentences for each test.
4. **Observe the follow-up-turn behavior.** In the `sub_agents` pattern, after the first turn transfers to a specialist, send a follow-up in the same session. Does it go to the specialist or back to the coordinator? Inspect the session's event stream.

# Key Takeaways

- **`sub_agents` is the transfer pattern.** Coordinator's LLM emits `transfer_to_agent(name=...)`; ADK routes; the specialist owns the response. Good for topic-shifts and extended specialist dialogs.
- **`AgentTool` is the consultant pattern.** Coordinator calls a specialist like a function; the coordinator composes the final reply. Good for specialists with clean I/O contracts whose work is part of a larger answer.
- **The child's `description=`** is the routing schema the coordinator's LLM reads. Write descriptions for the model to read.
- **Tell-tale sign:** final-response author = specialist → transfer; coordinator → consultant.
- **Interlude — multi-agent decomposition.** There's a coordination tax: extra calls, routing errors, context fragmentation. Decompose only when reuse, model-heterogeneity, or instruction-scale tests pass.
- **Default:** one agent with tools. Multi-agent is a deliberate choice, not a habit.

# Next up — M07: Callbacks as middleware

Six lifecycle hooks that wrap every invocation. `before_agent`, `after_agent`, `before_model`, `after_model`, `before_tool`, `after_tool`. Return `None` and things proceed normally; return a response object and you skip the LLM or tool altogether. It's Django middleware, Express middleware, Plugin hooks — for agents. The blocklist guardrail demo is the small-and-visual wow. See you there.